# Analyze Authoritative Merged BERTopic Artefacts

This is the fast analysis notebook. It should only be used after `Merged_BERTopic_All_Outlets.ipynb` has been rerun and the resulting artefacts have been marked as the authoritative final run.

Important naming note: the labels on the UMAP refer to **display-ranked merged topics**, not the raw BERTopic topic ids. For example, `Topic 2 [raw 33]` means: second-largest topic after the final merged ranking, with original raw BERTopic id `33`.


In [ ]:
import os
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing .git")


PROJECT_ROOT = find_project_root(Path.cwd())
MODULE_ROOT = PROJECT_ROOT / "1a_BERTopic"
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

os.environ["MPLCONFIGDIR"] = str(PROJECT_ROOT / ".mplconfig")

TOPIC_EXPORT_PATH = PROJECT_ROOT / "data" / "processed" / "df_combined_with_topic.csv"
TOPIC_OUTLET_TABLE_PATH = PROJECT_ROOT / "data" / "processed" / "merged_topic_outlet_counts.csv"
DISTORTION_SCORECARD_PATH = PROJECT_ROOT / "data" / "processed" / "merged_agenda_distortion_scores.csv"

In [ ]:
import importlib

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

import merged_outlets_analysis as moa
moa = importlib.reload(moa)

build_topic_outlet_frequency_table = moa.build_topic_outlet_frequency_table
build_agenda_distortion_scorecard = moa.build_agenda_distortion_scorecard
load_merged_analysis_cache = moa.load_merged_analysis_cache
load_saved_merged_model = moa.load_saved_merged_model
plot_outlet_colored_topic_umap = moa.plot_outlet_colored_topic_umap
validate_merged_analysis_cache = moa.validate_merged_analysis_cache


In [ ]:
merged_model, merged_model_path = load_saved_merged_model(PROJECT_ROOT)
merged_articles, merged_topic_info_display, merged_cache_metadata = load_merged_analysis_cache(PROJECT_ROOT)
cache_issues = validate_merged_analysis_cache(
    merged_articles,
    merged_topic_info_display,
    merged_cache_metadata,
)
if cache_issues:
    raise ValueError("\n".join(cache_issues))
if not merged_cache_metadata.get("authoritative_final_run", False):
    raise ValueError("The cached merged artefacts are not marked as the authoritative final run. Rebuild them in Merged_BERTopic_All_Outlets.ipynb.")

article_topics = pd.read_csv(TOPIC_EXPORT_PATH)

print(f"Authoritative run label: {merged_cache_metadata.get('run_label', 'missing')}")
print(f"Saved merged model: {merged_model_path}")
print("Cache generated at (UTC):", merged_cache_metadata["generated_at_utc"])
print("Merge similarity threshold:", merged_cache_metadata.get("min_similarity", "missing"))
print("Cached merged articles:", len(merged_articles))
print("Cached merged topic rows:", len(merged_topic_info_display))
print("Topic-info counts sum:", int(merged_topic_info_display["Count"].sum()))
print("df_combined + Topic rows:", len(article_topics))
print("Rows with final Topic assigned:", int(article_topics["Topic"].notna().sum()))
print("Rows with raw outlier Topic = -1:", int((merged_articles["merged_topic"] == -1).sum()))

display(merged_topic_info_display[["Topic", "Count", "DisplayTopic", "DisplayLabel"]].head(15))

In [ ]:
topic_outlet_table = build_topic_outlet_frequency_table(
    merged_articles,
    merged_topic_info_display,
    include_outliers=False,
)
topic_outlet_table.to_csv(TOPIC_OUTLET_TABLE_PATH, index=False)

display(topic_outlet_table)
print(f"Saved topic-by-outlet table to: {TOPIC_OUTLET_TABLE_PATH}")


In [ ]:
fig, ax = plot_outlet_colored_topic_umap(
    merged_articles,
    merged_topic_info_display,
    top_n=10,
    include_raw_topic_id_in_labels=True,
)
plt.show()


In [ ]:
distortion_scorecard = build_agenda_distortion_scorecard(
    merged_articles,
    merged_topic_info_display,
    reference_outlet_key="tagesschau",
    top_k=10,
    min_topic_articles=10,
)
distortion_scorecard.to_csv(DISTORTION_SCORECARD_PATH, index=False)

display(
    distortion_scorecard.round(
        {
            "Outlier_Rate": 3,
            "Entropy": 3,
            "JSD_vs_Tagesschau": 3,
            "Spearman_vs_Tagesschau": 3,
            "Top10_Overlap_vs_Tagesschau": 3,
            "Expected_Topics_ge_10": 1,
            "Coverage_Breadth": 3,
        }
    )
)
print(f"Saved agenda-distortion scorecard to: {DISTORTION_SCORECARD_PATH}")
